# Phase 1: Automated Smart Grid Dataset Discovery
### Final-Year Major Project: AI-Based Intrusion Detection in Smart Grids

This notebook automatically inspects the project directory, discovers all smart grid datasets, parses metadata, reads testbed documentation (`.docx`, `.xlsx`, `.md`), and compiles the Master Dataset Registry.

**Google Colab Instructions:**
Uncomment the Colab setup cell below if running on Google Colab.


In [ ]:
# --- Google Colab Setup (Uncomment if running on Colab) ---
# !pip install -q pandas numpy scikit-learn openpyxl python-docx pyyaml
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/MAJOR PROJECT


In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np

# Working directory resolution (works locally and in Colab)
if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

print(f"Project root set to: {PROJECT_ROOT}")


## 1. Directory Tree & Dataset File Discovery
Search recursively for dataset files (CSV, DOCX, XLSX, JSON) across the project folder.


In [ ]:
found_files = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    # Skip virtual environments and hidden git folders
    if any(p in root for p in ['.venv', '.git', '__pycache__', '.system_generated']):
        continue
    for f in files:
        if f.endswith(('.csv', '.docx', '.xlsx', '.json')):
            rel_p = os.path.relpath(os.path.join(root, f), PROJECT_ROOT)
            size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
            found_files.append({'Relative_Path': rel_p, 'Size_MB': round(size_mb, 2)})

df_files = pd.DataFrame(found_files)
print(f"Total relevant dataset/doc files discovered: {len(df_files)}")
df_files.head(15)


## 2. Load & Display the Master Dataset Registry
The automated registry categorizes all 6 discovered datasets across domain features (physical vs. cyber), rows, columns, and task feasibility.


In [ ]:
reg_path = os.path.join(PROJECT_ROOT, "reports", "dataset_registry.csv")
if os.path.exists(reg_path):
    df_reg = pd.read_csv(reg_path)
    display(df_reg[['Dataset_ID', 'Dataset_Name', 'Rows', 'Columns', 'Number_of_Classes', 'Temporal_Data', 'Binary_Task_Available', 'Multiclass_Task_Available', 'Anomaly_Task_Available', 'Unseen_Attack_Task_Available']])
else:
    print("Run src/generate_registry_and_reports.py to generate dataset_registry.csv")


## 3. Formatted Markdown Registry Details


In [ ]:
from IPython.display import Markdown, display
md_path = os.path.join(PROJECT_ROOT, "reports", "dataset_registry.md")
if os.path.exists(md_path):
    with open(md_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
